# Analysis: multiple seeds, late fusion and the B2/C1 boundary

Reads the probabilities saved by `01_multiseed_training`.

1. Late fusion: a logistic regression that combines the predictions of
   ModernBERT Only and the vocabulary-only model.
2. Mean and SD across seeds for every condition, with 95% CIs for F1 and QWK.
3. Paired comparison of predictions: each vocabulary condition vs ModernBERT Only,
   using a paired bootstrap over texts (difference in macro F1 and C1 recall,
   averaged over seeds, with a 95% CI).
4. B2 and C1 recall, and confusion matrices for ModernBERT Only, ModernBERT + AW,
   Vocabulary Only (AW) and Late Fusion (AW).

`STAGE` must match the stage `01_multiseed_training` was run with:

- `"val"`: results on the validation set. Late fusion is fitted with 5-fold
  cross-validation within the validation set.
- `"final"`: results on the test set. Late fusion is fitted on the whole
  validation set and applied to the test set.

In [4]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

STAGE = "final"
EVAL = "val" if STAGE == "val" else "test"

OUT_DIR = Path("robustness_outputs")
PROB_DIR = OUT_DIR / f"probs_{STAGE}"
cfg = json.load(open(OUT_DIR / f"run_config_{STAGE}.json"))
gold = np.load(OUT_DIR / f"gold_{STAGE}.npz")
Y = {sp: gold[f"{sp}_labels"] for sp in cfg["splits"]}

LABELS = ["A2", "B1", "B2", "C1"]
K = len(LABELS)
SEEDS = cfg["seeds"]
N_BOOT = 5000

BASELINE = "hf_only"
CONDITIONS = ["hf_only", "hf_AW_raw", "hf_AW_std", "hf_LazyA1_raw", "hf_LazyA1_std",
              "late_AW", "late_LazyA1", "vocab_AW", "vocab_LazyA1"]
COMPARED = [c for c in CONDITIONS if c != BASELINE and not c.startswith("vocab")]

trained = [c for c in CONDITIONS if not c.startswith("late")]
probs = {c: {s: dict(np.load(PROB_DIR / f"{c}_seed{s}.npz")) for s in SEEDS} for c in trained}
print(f"Evaluating on the {EVAL} set with {len(SEEDS)} seeds")

Evaluating on the test set with 10 seeds


In [5]:
# ---- Metrics ----------------------------------------------------------------
def confmat(y, p):
    return np.bincount(y * K + p, minlength=K * K).reshape(K, K)

def per_class(y, p):
    cm = confmat(y, p)
    tp = np.diag(cm).astype(float)
    prec = np.divide(tp, cm.sum(0), out=np.zeros(K), where=cm.sum(0) > 0)
    rec = np.divide(tp, cm.sum(1), out=np.zeros(K), where=cm.sum(1) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(K), where=(prec + rec) > 0)
    return prec, rec, f1

def qwk(y, p):
    cm = confmat(y, p).astype(float)
    w = (np.subtract.outer(np.arange(K), np.arange(K)) ** 2) / (K - 1) ** 2
    expected = np.outer(cm.sum(1), cm.sum(0)) / cm.sum()
    return 1 - (w * cm).sum() / (w * expected).sum()

METRICS = {
    "Precision": lambda y, p: per_class(y, p)[0].mean(),
    "Recall":    lambda y, p: per_class(y, p)[1].mean(),
    "F1":        lambda y, p: per_class(y, p)[2].mean(),
    "QWK":       qwk,
    "B2_recall": lambda y, p: per_class(y, p)[1][2],
    "C1_recall": lambda y, p: per_class(y, p)[1][3],
}

In [6]:
# ---- Late fusion --------------------------------------------------------------
# Logistic regression on the log-probabilities of ModernBERT only and the
# vocabulary-only model (same seed), fitted on the validation set.
def features(pb, pv):
    return np.hstack([np.log(pb + 1e-9), np.log(pv + 1e-9)])

def late_fusion(seed, mode):
    pb, pv, yv = probs[BASELINE][seed], probs[f"vocab_{mode}"][seed], Y["val"]
    if STAGE == "val":
        out = np.zeros_like(pb["val"])
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(pb["val"], yv):
            lr = LogisticRegression(max_iter=2000).fit(features(pb["val"][tr], pv["val"][tr]), yv[tr])
            out[te] = lr.predict_proba(features(pb["val"][te], pv["val"][te]))
        return {"val": out}
    lr = LogisticRegression(max_iter=2000).fit(features(pb["val"], pv["val"]), yv)
    return {"test": lr.predict_proba(features(pb["test"], pv["test"]))}

for mode in ["AW", "LazyA1"]:
    probs[f"late_{mode}"] = {s: late_fusion(s, mode) for s in SEEDS}

y = Y[EVAL]
preds = {c: {s: probs[c][s][EVAL].argmax(1) for s in SEEDS} for c in CONDITIONS}

In [7]:
# ---- 1. Mean, SD and 95% CI across seeds --------------------------------------
scores = pd.DataFrame([{"condition": c, "seed": s, **{m: f(y, preds[c][s]) for m, f in METRICS.items()}}
                       for c in CONDITIONS for s in SEEDS])
scores.to_csv(OUT_DIR / f"{STAGE}_scores_per_seed.csv", index=False)

rows = []
for c in CONDITIONS:
    for m in ["F1", "QWK"]:
        x = scores.loc[scores.condition == c, m]
        half = stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))
        rows.append({"condition": c, "metric": m, "mean": x.mean(), "SD": x.std(ddof=1),
                     "CI_low": x.mean() - half, "CI_high": x.mean() + half})
summary = pd.DataFrame(rows).round(3)
summary.to_csv(OUT_DIR / f"{STAGE}_summary_ci.csv", index=False)
summary

,condition,metric,mean,SD,CI_low,CI_high
0,hf_only,F1,0.686,0.024,0.669,0.702
1,hf_only,QWK,0.807,0.013,0.798,0.816
2,hf_AW_raw,F1,0.682,0.017,0.669,0.694
3,hf_AW_raw,QWK,0.803,0.009,0.797,0.810
4,hf_AW_std,F1,0.684,0.016,0.672,0.696
5,hf_AW_std,QWK,0.806,0.007,0.801,0.811
6,hf_LazyA1_raw,F1,0.688,0.012,0.679,0.696
7,hf_LazyA1_raw,QWK,0.808,0.009,0.801,0.814
8,hf_LazyA1_std,F1,0.673,0.025,0.655,0.691
9,hf_LazyA1_std,QWK,0.800,0.014,0.790,0.810


In [8]:
# Table in the format of Table 2: mean (SD) across seeds
fmt = lambda v: f"{v:.2f}".replace("0.", ".", 1)
agg = scores.groupby("condition")[["Precision", "Recall", "F1", "QWK"]].agg(["mean", "std"])
table = pd.DataFrame({m: agg[m]["mean"].map(fmt) + " (" + agg[m]["std"].map(fmt) + ")"
                      for m in ["Precision", "Recall", "F1", "QWK"]}).reindex(CONDITIONS)
table.to_csv(OUT_DIR / f"{STAGE}_table_mean_sd.csv")
table

,Precision,Recall,F1,QWK
condition,,,,
hf_only,.72 (.02),.68 (.02),.69 (.02),.81 (.01)
hf_AW_raw,.71 (.02),.67 (.02),.68 (.02),.80 (.01)
hf_AW_std,.71 (.01),.68 (.02),.68 (.02),.81 (.01)
hf_LazyA1_raw,.71 (.01),.68 (.01),.69 (.01),.81 (.01)
hf_LazyA1_std,.70 (.02),.66 (.03),.67 (.03),.80 (.01)
late_AW,.70 (.03),.69 (.03),.69 (.03),.82 (.02)
late_LazyA1,.70 (.02),.69 (.03),.69 (.02),.82 (.01)
vocab_AW,.48 (.01),.47 (.02),.47 (.01),.58 (.02)
vocab_LazyA1,.49 (.02),.47 (.03),.47 (.02),.56 (.04)


In [9]:
# ---- 2. Paired comparison with ModernBERT only ---------------------------------
# Texts are resampled with replacement; for each resample the difference from
# ModernBERT only is computed for every seed and averaged over seeds. The 2.5th
# and 97.5th percentiles give the 95% CI of the difference. A CI that includes
# zero means no reliable difference.
rng = np.random.default_rng(0)
boot_idx = rng.integers(0, len(y), size=(N_BOOT, len(y)))

rows = []
for c in COMPARED:
    for m in ["F1", "C1_recall"]:
        f = METRICS[m]
        diff = np.mean([f(y, preds[c][s]) - f(y, preds[BASELINE][s]) for s in SEEDS])
        boot = np.array([np.mean([f(y[i], preds[c][s][i]) - f(y[i], preds[BASELINE][s][i]) for s in SEEDS])
                         for i in boot_idx])
        rows.append({"condition": c, "metric": m, "difference": diff,
                     "CI_low": np.percentile(boot, 2.5), "CI_high": np.percentile(boot, 97.5)})

comparisons = pd.DataFrame(rows).round(3)
comparisons.to_csv(OUT_DIR / f"{STAGE}_paired_comparisons.csv", index=False)
comparisons

,condition,metric,difference,CI_low,CI_high
0,hf_AW_raw,F1,-0.004,-0.016,0.009
1,hf_AW_raw,C1_recall,0.015,-0.024,0.053
2,hf_AW_std,F1,-0.002,-0.014,0.011
3,hf_AW_std,C1_recall,0.012,-0.027,0.050
4,hf_LazyA1_raw,F1,0.002,-0.010,0.014
5,hf_LazyA1_raw,C1_recall,0.031,-0.007,0.068
6,hf_LazyA1_std,F1,-0.013,-0.026,0.000
7,hf_LazyA1_std,C1_recall,-0.013,-0.054,0.026
8,late_AW,F1,0.009,-0.016,0.034
9,late_AW,C1_recall,0.010,-0.055,0.075


In [10]:
# ---- 4. B2/C1 boundary: recall per level (mean (SD) across seeds) --------------
recall_tbl = pd.DataFrame({m: scores.groupby("condition")[m].mean().map(fmt) + " ("
                              + scores.groupby("condition")[m].std().map(fmt) + ")"
                           for m in ["B2_recall", "C1_recall"]}).reindex(CONDITIONS)
recall_tbl

,B2_recall,C1_recall
condition,,
hf_only,.81 (.03),.55 (.06)
hf_AW_raw,.80 (.02),.57 (.06)
hf_AW_std,.80 (.02),.57 (.08)
hf_LazyA1_raw,.80 (.03),.59 (.06)
hf_LazyA1_std,.80 (.02),.54 (.09)
late_AW,.66 (.06),.56 (.09)
late_LazyA1,.66 (.04),.56 (.08)
vocab_AW,.55 (.04),.36 (.10)
vocab_LazyA1,.54 (.10),.29 (.09)


In [11]:
# ---- 4. Confusion matrices (mean over seeds) -----------------------------------
def plot_cm(c, title):
    cm = np.mean([confmat(y, preds[c][s]) for s in SEEDS], axis=0)
    cm_norm = cm / cm.sum(1, keepdims=True)
    ylabels = [f"{l} (n={n})" for l, n in zip(LABELS, np.bincount(y, minlength=K))]
    text = [[f"{cm_norm[i, j] * 100:.1f}%" for j in range(K)] for i in range(K)]
    fig = go.Figure(go.Heatmap(z=cm_norm, x=LABELS, y=ylabels, colorscale=[[0, "white"], [1, "black"]],
                               text=text, texttemplate="%{text}", textfont={"size": 20}, showscale=False))
    fig.update_layout(title=f"{title}<br><sup>{EVAL} set, mean over {len(SEEDS)} seeds</sup>",
                      xaxis=dict(title="Predicted", tickfont=dict(size=18), title_font=dict(size=18)),
                      yaxis=dict(title="True", tickfont=dict(size=18), title_font=dict(size=18),
                                 autorange="reversed"),
                      width=550, height=500)
    fig.show()

plot_cm("hf_only", "ModernBERT Only")
plot_cm("hf_AW_raw", "ModernBERT + AW")
plot_cm("late_AW", "Late Fusion (AW)")

In [19]:
from plotly.subplots import make_subplots

def plot_cm_grid(conds, titles, ncols=2):
    nrows = int(np.ceil(len(conds) / ncols))
    fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=titles,
                        horizontal_spacing=0.06, vertical_spacing=0.08)
    ylabels = [f"{l} (n={n})" for l, n in zip(LABELS, np.bincount(y, minlength=K))]
    for k, c in enumerate(conds):
        r, col = k // ncols + 1, k % ncols + 1
        cm = np.mean([confmat(y, preds[c][s]) for s in SEEDS], axis=0)
        cm_norm = cm / cm.sum(1, keepdims=True)
        text = [[f"{cm_norm[i, j] * 100:.1f}%" for j in range(K)] for i in range(K)]
        fig.add_trace(go.Heatmap(z=cm_norm, x=LABELS, y=ylabels, zmin=0, zmax=1,
                                 colorscale=[[0, "white"], [1, "black"]], showscale=False,
                                 text=text, texttemplate="%{text}", textfont={"size": 16}), row=r, col=col)
        fig.update_xaxes(title_text="Predicted" if r == nrows else None, tickfont=dict(size=16),
                         showticklabels=(r == nrows), row=r, col=col)
        fig.update_yaxes(title_text="True" if col == 1 else None, autorange="reversed",
                         tickfont=dict(size=16), showticklabels=(col == 1), row=r, col=col)
    fig.update_annotations(font_size=18)
    fig.update_layout(width=950, height=850, font=dict(color="black"),
                      margin=dict(t=50, b=60, l=60, r=20))
    fig.write_image(OUT_DIR / f"{STAGE}_confusion_matrices.png", scale=6)
    fig.show()

plot_cm_grid(["hf_only", "hf_AW_raw", "vocab_AW", "late_AW"],
             ["ModernBERT Only", "ModernBERT + AW", "Vocabulary Only (AW)", "Late Fusion (AW)"])